In [ ]:
#  Resume Screening Analysis

In [ ]:
# ==============================
# Import Required Libraries
# ==============================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_squared_error,
    r2_score,
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_curve,
    auc
)


In [ ]:
# ============================================
# Load Dataset
# ============================================

df = pd.read_csv("data/AI_Resume_Screening.csv")

# Display first 5 rows
df.head()

In [ ]:
# ============================================
# Data Cleaning
# ============================================

# Fill missing certifications
df['Certifications'] = df['Certifications'].fillna('No Certifications')

# Encode Recruiter Decision (Reject = 0, Hire = 1)
df['Recruiter_Decision_Encoded'] = df['Recruiter Decision'].replace({
    'Reject': 0,
    'Hire': 1
})

# Check missing values
print(df.isnull().sum())

# Save cleaned dataset
df.to_csv("data/AI_Resume_Screening_Cleaned.csv", index=False)

print("✅ Data cleaning completed and cleaned file saved.")

In [ ]:
## Exploratory Data Analysis

In [ ]:
# AI Score Distribution

plt.figure(figsize=(6,4))
sns.histplot(df['AI Score (0-100)'], bins=20)
plt.title("Distribution of AI Scores")
plt.xlabel("AI Score")
plt.ylabel("Count")
plt.show()

In [ ]:
# Experience vs AI Score

plt.figure(figsize=(6,4))
sns.scatterplot(x='Experience (Years)', 
                y='AI Score (0-100)', 
                data=df)

plt.title("Experience vs AI Score")
plt.show()

In [ ]:
# Correlation Heatmap

correlation = df[[
    "Experience (Years)",
    "AI Score (0-100)",
    "Projects Count",
    "Salary Expectation ($)",
    "Recruiter_Decision_Encoded"
]].corr()

plt.figure(figsize=(6,4))
sns.heatmap(correlation, annot=True, cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
## Linear Regression Model: Predict AI Score based on candidate features

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Define Features and Target
X_lr = df[[
    "Experience (Years)",
    "Projects Count",
    "Salary Expectation ($)"
]]

y_lr = df["AI Score (0-100)"]

# Train-Test Split
X_train_lr, X_test_lr, y_train_lr, y_test_lr = train_test_split(
    X_lr, y_lr, test_size=0.3, random_state=42
)

# Create Model
lr_model = LinearRegression()

# Train Model
lr_model.fit(X_train_lr, y_train_lr)

# Predictions
y_pred_lr = lr_model.predict(X_test_lr)

# Evaluation
mse = mean_squared_error(y_test_lr, y_pred_lr)
r2 = r2_score(y_test_lr, y_pred_lr)

print("Linear Regression Results")
print("--------------------------")
print("Mean Squared Error:", mse)
print("R2 Score:", r2)

In [ ]:
## Logistic Regression

In [ ]:
X = df[[
    "Experience (Years)",
    "Projects Count",
    "AI Score (0-100)",
    "Salary Expectation ($)"
]]

y = df["Recruiter_Decision_Encoded"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

model = LogisticRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

In [ ]:
# ROC Curve

fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6,4))
plt.plot(fpr, tpr, label=f"ROC Curve (AUC = {roc_auc:.2f})")
plt.plot([0,1], [0,1], 'k--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()

In [ ]:
## MySQL Database Connection

In [ ]:
# ============================================
# Connect to MYSQL Database
# ============================================

db = mysql.connector.connect(
    host="localhost",
    user="root",
    password="your_password",
    database="Resume"
)

if db.is_connected():
    print("✅ Connected to MySQL successfully")

db.close()

In [ ]:
# Insert Cleaned Data into MySQL

import csv
import mysql.connector

db = mysql.connector.connect(
    host="localhost",
    user="root",
    password="your_password",
    database="Resume"
)

cursor = db.cursor()

sql = """
INSERT INTO Resume_Screening
(Resume_ID, Name, Skills, `Experience (Years)`, Education,
 Certifications, `Job Role`, `Recruiter Decision`,
 `Salary Expectation ($)`, `Projects Count`,
 `AI Score (0-100)`, Recruiter_Decision_Encoded)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
"""

with open("data/AI_Resume_Screening_Cleaned.csv", newline="", encoding="utf-8") as file:
    reader = csv.reader(file)
    next(reader)

    for row in reader:
        cursor.execute(sql, row)

db.commit()
cursor.close()
db.close()

print("✅ CSV data inserted into MySQL successfully!")

In [ ]:
## Key Business Insights

- Higher AI scores significantly increase the probability of candidate selection.
- Candidates with more experience and higher project counts show stronger hiring rates.
- Salary expectations directly influence recruiter shortlisting decisions.
- AI Score, experience, and project count together are strong predictors of hiring outcomes.
- Data-driven resume screening reduces manual effort and improves recruitment efficiency.